# Downstream performance

Writes the downstream table and figure from the published `ward_intake` sweep on SynTabFall.

In [1]:
from itertools import groupby
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from figures import (
    CLASSIFIER,
    COLORS,
    COLUMN_WIDTH_PT,
    PAPER_STYLE,
    as_percent,
    measure_label,
    scenario_label,
    set_size,
)
from matplotlib.lines import Line2D

from experiments.measures import measure_dimension

RESULTS = Path("../measurements/downstream-performance")
OUT = Path("../results")
SCENARIO = "ward_intake"
RESPONSE = 0.05  # the data quality table's threshold for a change worth reporting

raw = pd.concat(pd.read_parquet(p) for p in sorted(RESULTS.glob("**/results.parquet")))
obs = raw[raw.status == "ok"].copy()
obs["column"] = obs["column"].fillna("(table)")
obs = obs.astype({"dq_score": float, "error_rate": float, "column_error_rate": float})

In [2]:
# The classifier's metrics belong to the fold and repeat on every measure row, so one row per fold.
FOLD_KEY = ["scenario", "error_rate", "seed", "fold"]
COUNTS = ["tp", "fp", "tn", "fn"]
CLASSIFIER_ROWS = {
    "auc": "__classifier__",
    "precision": "__classifier_precision__",
    "recall": "__classifier_recall__",
    "macro_f1": "__classifier_macro_f1__",
}


def add_rates(frame, side):
    """Precision, recall and macro-F1 (as sklearn's `f1_score(average="macro")`) from `<count>_<side>`."""
    tp, fp, tn, fn = (frame[f"{count}_{side}"] for count in COUNTS)
    frame[f"precision_{side}"] = tp / (tp + fp)
    frame[f"recall_{side}"] = tp / (tp + fn)
    frame[f"macro_f1_{side}"] = (2 * tp / (2 * tp + fp + fn) + 2 * tn / (2 * tn + fp + fn)) / 2
    return frame


folds = add_rates(add_rates(obs.drop_duplicates(FOLD_KEY).copy(), "clean"), "dirty")


def classifier_row(metric):
    """One classifier metric per rate: the clean test fold at 0%, the dirty one at each swept rate."""
    stacked = pd.concat(
        [
            folds[folds.scenario == "clean"].assign(error_rate=0.0, value=lambda t: t[f"{metric}_clean"]),
            folds[folds.scenario == SCENARIO].assign(value=lambda t: t[f"{metric}_dirty"]),
        ]
    )
    by_rate = stacked.groupby("error_rate")["value"].agg(["mean", "std", "count"]).reset_index()
    return by_rate.assign(measure=CLASSIFIER_ROWS[metric])


# The change in score, column by column: the same measure, column and fold, with errors minus without.
# Only the columns the scenario really corrupted count. A measure's level in a fold is its clean score over
# those columns plus the fold's mean change.
key = ["measure", "column", "fold"]
clean = obs[obs.scenario == "clean"]
damaged = obs[(obs.scenario == SCENARIO) & (obs.column_error_rate > 0)]
delta = damaged.assign(
    change=damaged.dq_score.to_numpy()
    - clean.set_index(key).dq_score.reindex(pd.MultiIndex.from_frame(damaged[key])).to_numpy()
)
level0 = clean[clean.column.isin(set(damaged.column))].groupby(["measure", "fold"]).dq_score.mean().rename("level0")
changed = (
    delta.groupby(["measure", "error_rate", "fold"]).change.mean().reset_index().join(level0, on=["measure", "fold"])
)
by_fold = pd.concat(
    [
        level0.reset_index().assign(error_rate=0.0, level=lambda t: t.level0),
        changed.assign(level=changed.level0 + changed.change),
    ]
)
per_rate = by_fold.groupby(["measure", "error_rate"])["level"].agg(["mean", "std", "count"]).reset_index()

CELLS = pd.concat([per_rate, *(classifier_row(metric) for metric in CLASSIFIER_ROWS)]).set_index(
    ["measure", "error_rate"]
)
RATES = sorted(per_rate.error_rate.unique())

## Downstream table

A cell is a measure's mean score over the columns `ward_intake` corrupted, ± the standard deviation over the
five folds. The 0% column is the same measure on clean data over those columns. The first four rows are
ROC-AUC, precision, recall and macro-F1, the last three at the threshold the runner tunes for macro-F1.
Δ is the change from 0% to 50%, bold at 0.05 or more; a dash is a measure that returned no score.

In [3]:
def cell_tex(measure, rate):
    """The mean over folds with the fold SD beside it; a dash where the measure returned no score."""
    if (measure, rate) not in CELLS.index or pd.isna(CELLS.loc[(measure, rate), "mean"]):
        return "--"
    row = CELLS.loc[(measure, rate)]
    return f"${row['mean']:.3f}${{\\scriptsize$\\pm{row['std']:.3f}$}}"


def delta_tex(measure):
    """The change from 0% to the highest rate, bold at `RESPONSE` or more."""
    ends = [(measure, RATES[0]), (measure, RATES[-1])]
    if any(e not in CELLS.index for e in ends) or CELLS.loc[ends, "mean"].isna().any():
        return "--"
    change = CELLS.loc[ends[1], "mean"] - CELLS.loc[ends[0], "mean"]
    body = f"{change:+.3f}" if round(change, 3) != 0 else "0.000"
    # `\textbf` does nothing inside math mode.
    return f"$\\mathbf{{{body}}}$" if abs(change) >= RESPONSE else f"${body}$"


def row_tex(label, measure):
    return f"    {label} & " + " & ".join(cell_tex(measure, r) for r in RATES) + f" & {delta_tex(measure)} " + r"\\"


pct = [f"{round(r * 100):g}\\%" for r in RATES]
caption = (
    f"    Mean DQ score over the columns \\texttt{{{scenario_label(SCENARIO)}}} damaged, across all "
    f"{int(CELLS['count'].max())} folds with the fold standard deviation beside it in a smaller font; "
    f"the first four rows are ROC-AUC, precision, recall and macro-$F_1$ at "
    f"the tuned threshold, $\\Delta$ is the change from ${pct[0]}$ to ${pct[-1]}$ (bold at ${RESPONSE:g}$ "
    f"or more), and a dash is a measure that returned no score."
)
lines = [
    r"\begin{table*}[t]",
    r"  \centering",
    r"  \footnotesize",
    r"  \setlength{\tabcolsep}{2pt}",
    r"  \caption{%",
    caption + "%",
    r"  }",
    r"  \label{tab:downstream-performance}",
    r"  \begin{tabular}{lrrrrrrr}",
    r"    \toprule",
    r"    & \multicolumn{6}{c}{Share of Erroneous Cells} & \\",
    r"    \cmidrule(lr){2-7}",
    "    Data Quality Measure & " + " & ".join(pct) + r" & $\Delta$ \\",
    r"    \midrule",
    row_tex("ROC-AUC", CLASSIFIER_ROWS["auc"]),
    row_tex("Precision", CLASSIFIER_ROWS["precision"]),
    row_tex("Recall", CLASSIFIER_ROWS["recall"]),
    row_tex("Macro-$F_1$", CLASSIFIER_ROWS["macro_f1"]),
    r"    \midrule",
]
# Grouped by ISO dimension, alphabetical within each group, as in the data quality table.
measures = sorted(per_rate.measure.unique(), key=lambda m: (measure_dimension(m), measure_label(m)))
for index, (_dimension, group) in enumerate(groupby(measures, key=measure_dimension)):
    if index:
        lines.append(r"    \addlinespace")
    lines += [row_tex(measure_label(m), m) for m in group]
lines += [r"    \bottomrule", r"  \end{tabular}", r"\end{table*}"]

(OUT / "downstream_table.tex").write_text("\n".join(lines) + "\n")
CELLS.reset_index().to_csv(OUT / "downstream_table.csv", index=False)
print((OUT / "downstream_table.tex").read_text())

\begin{table*}[t]
  \centering
  \footnotesize
  \setlength{\tabcolsep}{2pt}
  \caption{%
    Mean DQ score over the columns \texttt{ward intake} damaged, across all 5 folds with the fold standard deviation beside it in a smaller font; the first four rows are ROC-AUC, precision, recall and macro-$F_1$ at the tuned threshold, $\Delta$ is the change from $0\%$ to $50\%$ (bold at $0.05$ or more), and a dash is a measure that returned no score.%
  }
  \label{tab:downstream-performance}
  \begin{tabular}{lrrrrrrr}
    \toprule
    & \multicolumn{6}{c}{Share of Erroneous Cells} & \\
    \cmidrule(lr){2-7}
    Data Quality Measure & 0\% & 1\% & 5\% & 10\% & 25\% & 50\% & $\Delta$ \\
    \midrule
    ROC-AUC & $0.879${\scriptsize$\pm0.003$} & $0.878${\scriptsize$\pm0.004$} & $0.868${\scriptsize$\pm0.007$} & $0.856${\scriptsize$\pm0.010$} & $0.826${\scriptsize$\pm0.019$} & $0.776${\scriptsize$\pm0.036$} & $\mathbf{-0.103}$ \\
    Precision & $0.139${\scriptsize$\pm0.011$} & $0.138${\scriptsize$

## Downstream figure

Each black line is a measure's change in score from clean data; the purple line is the classifier's change in
ROC-AUC on its own right-hand axis, with the fold SD as whiskers. Top: measures that change by 0.05 or more.
Bottom: measures that change by less, but fall at every rate and by more than three fold SDs at 50%. Both axes
share their zero and tick heights. The figure fits one column: each panel's legend sits between its title and
the plot.

In [4]:
clean_level = CELLS.xs(RATES[0], level="error_rate")["mean"]
CHANGE = CELLS[["mean", "std"]].assign(
    change=lambda t: t["mean"] - clean_level.reindex(t.index.get_level_values("measure")).to_numpy()
)
DELTA = CHANGE.xs(RATES[-1], level="error_rate").change.dropna()
SD = CHANGE.xs(RATES[-1], level="error_rate")["std"]


def falls_at_every_rate(measure):
    return (CHANGE.loc[measure].change.sort_index().diff().dropna() < 0).all()


roster = DELTA.index.intersection(per_rate.measure.unique())
LARGE = sorted((m for m in roster if abs(DELTA[m]) >= RESPONSE), key=DELTA.get)
# Below the threshold but not noise: a fall at every rate, and by more than three fold SDs at the highest.
SMALL = sorted(
    (m for m in roster if 3 * SD[m] < abs(DELTA[m]) < RESPONSE and falls_at_every_rate(m)),
    key=DELTA.get,
)

AUC = CHANGE.loc[CLASSIFIER_ROWS["auc"]].sort_index()
auc = CELLS.loc[CLASSIFIER_ROWS["auc"], "mean"]
print(
    f"ROC-AUC {auc[RATES[0]]:.3f} → {auc[RATES[-1]]:.3f}; bottom panel at {RATES[-1]:.0%}: "
    + ", ".join(f"{m} {abs(DELTA[m]) / SD[m]:.1f} fold SDs" for m in SMALL)
)

MARKERS = ("x", "s", "^", "D", "v")  # every measure is black, so a marker tells them apart
RATE_TICKS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)
# Both axes of a panel fall `N_STEPS` round steps below zero with the same padding, so their ticks line up.
STEPS = (0.01, 0.02, 0.025, 0.05, 0.1, 0.2, 0.25, 0.5)
N_STEPS = 3
PAD_BELOW, PAD_ABOVE = 0.12, 0.10
FIGURE = "2026-09-18-downstream-performance.pdf"


def nice_step(depth):
    """The smallest round tick step whose `N_STEPS` steps reach `depth`."""
    return next(step for step in STEPS if N_STEPS * step >= depth)


def set_steps(ax, step):
    ax.set_ylim(-(N_STEPS + PAD_BELOW) * step, PAD_ABOVE * step)
    ax.set_yticks([-i * step for i in range(N_STEPS + 1)])


AUC_STEP = nice_step(float((AUC["std"] - AUC.change).max()))  # whiskers included


def draw_panel(ax, measures, title):
    """The measures on the left axis, ROC-AUC on a twinned right one, and a legend headed by `title` above."""
    step = nice_step(float(-DELTA[measures].min()))
    # The legend lists the lines in the order they end at the highest rate, ROC-AUC on its own axis's scale.
    ends = {measure: float(DELTA[measure]) for measure in measures}
    ends[CLASSIFIER_ROWS["auc"]] = float(DELTA[CLASSIFIER_ROWS["auc"]]) / AUC_STEP * step
    order = sorted(ends, key=ends.get, reverse=True)
    markers = dict(zip([m for m in order if m != CLASSIFIER_ROWS["auc"]], MARKERS, strict=False))

    handles = {}
    for measure in measures:
        rows = CHANGE.loc[measure].sort_index()
        (handles[measure],) = ax.plot(
            rows.index,
            rows.change,
            color=COLORS[SCENARIO],
            marker=markers[measure],
            lw=1.0,
            ms=3.0,
            mew=0.9,
            label=measure_label(measure),
        )
    set_steps(ax, step)
    ax.grid(axis="y", visible=True, alpha=0.3, lw=0.5)
    ax.grid(axis="x", visible=False)
    ax.set_xticks(RATE_TICKS, [as_percent(rate) for rate in RATE_TICKS])
    ax.set_xlim(-0.015, RATES[-1] + 0.015)
    ax.spines["bottom"].set_bounds(0, RATES[-1])

    twin = ax.twinx()
    twin.errorbar(
        AUC.index,
        AUC.change,
        yerr=AUC["std"],
        color=CLASSIFIER,
        marker="o",
        lw=1.0,
        ms=2.6,
        elinewidth=0.7,
        capsize=1.6,
        capthick=0.7,
        alpha=0.9,
    )
    set_steps(twin, AUC_STEP)
    twin.grid(visible=False)
    twin.spines["top"].set_visible(False)
    twin.spines["right"].set_visible(True)
    twin.spines["right"].set_color(CLASSIFIER)
    twin.tick_params(axis="y", colors=CLASSIFIER)
    # Lift the host above its twin, so the black lines sit over the purple.
    ax.set_zorder(twin.get_zorder() + 1)
    ax.patch.set_visible(False)

    handles[CLASSIFIER_ROWS["auc"]] = Line2D([], [], color=CLASSIFIER, marker="o", lw=1.0, ms=2.6, label="ROC-AUC")
    # Above the panel in two columns, since the column is too narrow to hold six entries inside it.
    ax.legend(
        handles=[handles[m] for m in order],
        ncol=2,
        title=title,
        title_fontsize=plt.rcParams["axes.titlesize"],
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        borderaxespad=0,
        columnspacing=1.0,
    )


with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(
        2,
        1,
        figsize=set_size(COLUMN_WIDTH_PT, subplots=(2, 1), height_ratio=0.9),
        sharex=True,
        layout="constrained",
    )
    # Matplotlib has no right-hand `supylabel`: keep a strip free at the right edge and centre the label in it,
    # at the height `supylabel` has on the left.
    strip = 1.4 * plt.rcParams["figure.labelsize"] / 72 / fig.get_figwidth()
    fig.get_layout_engine().set(w_pad=1 / 72, h_pad=1 / 72, hspace=0.04, rect=(0, 0, 1 - strip, 1))
    draw_panel(axes[0], LARGE, rf"DQ Score $|\Delta| \geq {RESPONSE:g}$")
    draw_panel(axes[1], SMALL, rf"DQ Score $|\Delta| < {RESPONSE:g}$")
    fig.supxlabel("Share of Erroneous Cells")
    fig.supylabel("Change in DQ Score")
    fig.text(
        1,
        0.5,
        "Change in ROC-AUC",
        rotation=90,
        ha="right",
        va="center",
        color=CLASSIFIER,
        fontsize=plt.rcParams["figure.labelsize"],
    )
    fig.savefig(OUT / FIGURE)
    plt.close(fig)

ROC-AUC 0.879 → 0.776; bottom panel at 50%: DataValueDistribution 6.5 fold SDs, SyntacticDataAccuracy 25.9 fold SDs, DataFormatConsistency 9.3 fold SDs
